# SSA SIRUP — Pull & Convert (Notebook)

Pull procurement records from Indonesia's **SIRUP Inaproc** for a given budget year (*tahun anggaran*), optionally filtered to a single ministry/agency (*KLDI*), then convert to **Feather** + pipe-delimited **CSV**.

Resume-safe: re-running the same year/KLDI continues from the last checkpoint.

**How to use:** edit the two values in the *Configuration* cells below, then *Run All*.

## 1. Configuration

Edit the value in **each of the next two cells**, then *Run All*:

- **ID_KLDI** — ministry/agency code (blank = all agencies)
- **TAHUN_ANGGARAN** — budget year (blank = current year)

In [ ]:
# ===== ID KLDI (ministry / agency CODE) =====
# The agency CODE (field `idKldi`), e.g. "L40" = Mahkamah Agung,
# "K9" = Kementerian Kesehatan, "K2" = Kementerian PUPR.
# NOTE: this is the CODE, not the agency name (the `kldi` field is the name).
# Leave BLANK ("") to pull/keep ALL agencies.
ID_KLDI = ""

In [ ]:
# ===== TAHUN ANGGARAN (budget year) =====
# The budget year to pull, e.g. "2026".
# Leave BLANK ("") to use the CURRENT year automatically.
TAHUN_ANGGARAN = ""

In [ ]:
# ===== resolved values (no need to edit) =====
from datetime import datetime

# True = pull JSONL only, skip Feather/CSV conversion.
NO_CONVERT = False

TAHUN = int(TAHUN_ANGGARAN) if TAHUN_ANGGARAN.strip() else datetime.now().year
KLDI = ID_KLDI.strip().upper()

_suffix = f"_{KLDI}" if KLDI else ""
OUTPUT_FILE = f"sirup_{TAHUN}{_suffix}_all.jsonl"
CHECKPOINT_FILE = f"sirup_{TAHUN}{_suffix}_checkpoint.json"
FEATHER_FILE = f"sirup_{TAHUN}{_suffix}_all.feather"
CSV_FILE = f"sirup_{TAHUN}{_suffix}_all.csv"

print(f"Tahun Anggaran : {TAHUN}" + ("  (current year)" if not TAHUN_ANGGARAN.strip() else ""))
print(f"idKldi filter  : {KLDI if KLDI else '(all agencies)'}")
print(f"JSONL          : {OUTPUT_FILE}")
print(f"Feather        : {FEATHER_FILE}")
print(f"CSV            : {CSV_FILE}")

## 2. Setup — imports, constants, helpers

In [ ]:
import json
import os
import sys
import time

import httpx

BASE_URL = "https://sirup.inaproc.id/sirup/caripaketctr/search"

PAGE_SIZE = 100_000
RETRY_DELAY = 15
REQUEST_DELAY = 0.5

# PLAY_SESSION cookie template — tahunAnggaranPilihan is injected from TAHUN.
# If pulls fail with session errors, refresh by visiting sirup.inaproc.id in a
# browser, copying the new PLAY_SESSION value, and replacing PLAY_SESSION below.
PLAY_SESSION = (
    "596baed944a2fbb195afe4ace1a0fbbbc3cfc38f"
    "-___TS=1784118415462&tahunAnggaranPilihan={tahun}"
    "&menu=cariPaket2&___ID=6ab5d171-a57e-46ef-8333-6fa4e48507b5"
)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:133.0) Gecko/20100101 Firefox/133.0",
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "Accept-Language": "en-US,en;q=0.5",
    "X-Requested-With": "XMLHttpRequest",
    "Referer": "https://sirup.inaproc.id/sirup/caripaketctr/index",
    "Connection": "keep-alive",
    "Sec-Fetch-Dest": "empty",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Site": "same-origin",
}

GA_COOKIES = {
    "_ga_KLDH3FQ7DR": "GS2.1.s1782194818$o3$g1$t1782194819$j59$l0$h0",
    "_ga": "GA1.1.933612520.1782181854",
    "_ga_3BPJ11C32J": "GS2.1.s1782194820$o2$g0$t1784194820$j60$l0$h0",
    "_ga_D78WKTMJC6": "GS2.1.s1784194820$o3$g1$t1784116617$j60$l0$h0",
}

In [ ]:
def build_cookies(tahun):
    cookies = dict(GA_COOKIES)
    cookies["PLAY_SESSION"] = PLAY_SESSION.format(tahun=tahun)
    return cookies


def build_params(tahun, start, draw):
    # NOTE on KLDI filtering:
    #   Each record has TWO fields: `kldi` = agency NAME (e.g. "Mahkamah Agung"),
    #   `idKldi` = agency CODE (e.g. "L40"). The server's `kldi` URL param filters
    #   by the NAME, so it can't filter by code. We therefore ALWAYS pull all
    #   agencies here (kldi="") and filter by `idKldi` client-side in pull().
    params = {
        "tahunAnggaran": str(tahun),
        "jenisPengadaan": "",
        "metodePengadaan": "",
        "minPagu": "",
        "maxPagu": "",
        "bulan": "",
        "lokasi": "",
        "kldi": "",                   # always pull all; filter client-side by idKldi
        "pdn": "",
        "ukm": "",
        "draw": str(draw),
        "start": str(start),
        "length": str(PAGE_SIZE),
        "order[0][column]": "5",
        "order[0][dir]": "DESC",
        "search[value]": "",
        "search[regex]": "false",
        "_": str(int(time.time() * 1000)),
    }
    col_data = ["", "paket", "pagu", "jenisPengadaan", "isPDN", "isUMK",
                "metode", "pemilihan", "kldi", "satuanKerja", "lokasi", "id"]
    for i, cd in enumerate(col_data):
        searchable = "true" if cd else "false"
        orderable = "true" if cd else "false"
        params[f"columns[{i}][data]"] = cd
        params[f"columns[{i}][name]"] = ""
        params[f"columns[{i}][searchable]"] = searchable
        params[f"columns[{i}][orderable]"] = orderable
        params[f"columns[{i}][search][value]"] = ""
        params[f"columns[{i}][search][regex]"] = "false"
    return params


def filter_by_kldi(records, kldi):
    """Keep only records whose idKldi == kldi (case-insensitive). kldi='' keeps all."""
    if not kldi:
        return records
    k = kldi.upper()
    return [r for r in records if str(r.get("idKldi", "")).upper() == k]

In [ ]:
def count_jsonl_lines(filepath):
    if not os.path.exists(filepath):
        return 0
    count = 0
    with open(filepath, "r", encoding="utf-8") as f:
        for _ in f:
            count += 1
    return count


def load_checkpoint(checkpoint_file):
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, "r") as f:
            return json.load(f)
    return None


def save_checkpoint(checkpoint_file, start, saved_count, total):
    with open(checkpoint_file, "w") as f:
        json.dump({"start": start, "saved_count": saved_count, "total": total}, f, indent=2)


def append_records(output_file, records):
    with open(output_file, "a", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def _print_failure(resp, start, attempt):
    """Dump everything useful about a failed response so 403/4xx is diagnosable."""
    print(f"  [!] Attempt {attempt+1}/5 failed at offset {start}: "
          f"HTTP {resp.status_code} {resp.reason_phrase}")
    interesting = ["server", "cf-ray", "x-frame-options", "content-type",
                   "set-cookie", "location", "x-cache", "via"]
    hdrs = {k: v for k, v in resp.headers.items() if k.lower() in interesting}
    if hdrs:
        print(f"      headers: {hdrs}")
    body = (resp.text or "").strip()
    if body:
        print(f"      body[:500]: {body[:500]!r}")
    else:
        print("      body: (empty)")


def fetch_page(client, tahun, start, draw):
    params = build_params(tahun, start, draw)
    for attempt in range(5):
        try:
            resp = client.get(BASE_URL, params=params)
            if resp.status_code >= 400:
                _print_failure(resp, start, attempt)
                if resp.status_code in (401, 403):
                    print("  [HINT] 403/401 usually means: (a) the PLAY_SESSION cookie "
                          "expired, or (b) the server is blocking this IP "
                          "(e.g. Colab's US egress vs. an Indonesia-only site). "
                          "Refresh the cookie from a fresh browser session; if it "
                          "still 403s from Colab, the IP is blocked — run locally "
                          "or via an Indonesia-based proxy.")
                    raise httpx.HTTPStatusError(
                        f"HTTP {resp.status_code}", request=resp.request, response=resp)
            resp.raise_for_status()
            return resp.json()
        except (httpx.HTTPError, json.JSONDecodeError):
            if attempt < 4:
                time.sleep(RETRY_DELAY * (attempt + 1))
            else:
                raise

### 2b. (Optional) Diagnose connectivity — run if you get 403

This checks (a) what IP/region Colab is egressing from, and (b) whether the SIRUP **homepage** loads at all. If the homepage also 403s, your Colab IP is blocked (geo/WAF) and you must run locally or via an Indonesia proxy. If the homepage loads but `/search` 403s, it's a session/cookie problem.

In [ ]:
# Diagnose: egress IP + whether the SIRUP homepage and the AJAX search respond.
try:
    info = httpx.get("https://ipinfo.io/json", timeout=20).json()
    print(f"[NET] egress IP: {info.get('ip')}  region: {info.get('country')} / {info.get('region')}")
except Exception as e:
    print(f"[NET] could not fetch ipinfo: {e}")

try:
    r = httpx.get("https://sirup.inaproc.id/sirup/caripaketctr/index",
                  headers=HEADERS, timeout=30, follow_redirects=True)
    print(f"[HOME] /index -> HTTP {r.status_code}  ({len(r.text)} bytes)")
except Exception as e:
    print(f"[HOME] /index failed: {e}")

try:
    r = httpx.get(BASE_URL, params=build_params(TAHUN, 0, 1),
                  headers=HEADERS, cookies=build_cookies(TAHUN),
                  timeout=30, follow_redirects=True)
    print(f"[AJAX] /search -> HTTP {r.status_code}")
    print(f"[AJAX] body[:300]: {r.text[:300]!r}")
except Exception as e:
    print(f"[AJAX] /search failed: {e}")

## 3. Pull records

In [ ]:
def pull(tahun, kldi, output_file, checkpoint_file):
    """Pull all records for the given budget year into output_file (JSONL).

    The server cannot filter by agency code (its `kldi` param matches the agency
    NAME), so we always pull ALL agencies and filter by `idKldi` client-side when
    `kldi` is set. Note: a KLDI-specific pull therefore pages through the entire
    year; if you already have the full-year Feather, filter it directly instead
    (see the "Filter existing Feather" cell below)."""
    checkpoint = load_checkpoint(checkpoint_file)
    total_records = 0
    saved_count = 0   # records actually written (after KLDI filter)
    seen_count = 0    # records scanned from server

    if checkpoint:
        saved_count = count_jsonl_lines(output_file)
        start_offset = checkpoint["start"]
        total_records = checkpoint["total"]
        seen_count = checkpoint.get("seen", start_offset)
        draw = start_offset // PAGE_SIZE + 1
        print(f"[RESUME] {saved_count:,} kept records, starting at offset {start_offset:,}")
    else:
        start_offset = 0
        draw = 1
        if os.path.exists(output_file):
            print(f"[WARN] {output_file} exists but no checkpoint. Starting fresh.")
            os.remove(output_file)

    client = httpx.Client(
        headers=HEADERS,
        cookies=build_cookies(tahun),
        timeout=120.0,
        follow_redirects=True,
    )

    try:
        label = f"tahunAnggaran={tahun}" + (f", idKldi={kldi}" if kldi else ", idKldi=all")
        print(f"[INFO] Fetching initial page to get total count ({label})...")
        result = fetch_page(client, tahun, start_offset, draw)

        if not checkpoint:
            total_records = result.get("recordsTotal", 0)
            records_filtered = result.get("recordsFiltered", 0)
            print(f"   Server total:       {total_records:,}")
            print(f"   Server filtered:    {records_filtered:,}")
            print(f"   KLDI filter:        {kldi if kldi else '(none - keeping all)'}")
            print(f"   Pages:              {(total_records + PAGE_SIZE - 1) // PAGE_SIZE:,}")
            print(f"   Page size:          {PAGE_SIZE:,}")
            print()

        page_data = result.get("data", [])
        if not page_data and start_offset == 0:
            print("[ERROR] No data returned on first request! Check cookies/auth.")
            print(f"   Response keys: {list(result.keys())}")
            return False

        kept = filter_by_kldi(page_data, kldi)
        seen_count += len(page_data)
        append_records(output_file, kept)
        saved_count += len(kept)
        start_offset += PAGE_SIZE
        draw += 1
        save_checkpoint(checkpoint_file, start_offset, saved_count, total_records)

        pct = min(100.0, start_offset / total_records * 100) if total_records else 0
        print(f"  Page {draw-1}: scanned {len(page_data):,}, kept {len(kept):,} "
              f"| kept {saved_count:,} | scanned {seen_count:,}/{total_records:,} ({pct:.1f}%)")

        while start_offset < total_records:
            time.sleep(REQUEST_DELAY)
            result = fetch_page(client, tahun, start_offset, draw)
            page_data = result.get("data", [])

            if not page_data:
                print(f"  [WARN] No data at offset {start_offset:,}, stopping.")
                break

            kept = filter_by_kldi(page_data, kldi)
            seen_count += len(page_data)
            append_records(output_file, kept)
            saved_count += len(kept)
            start_offset += PAGE_SIZE
            draw += 1
            save_checkpoint(checkpoint_file, start_offset, saved_count, total_records)

            pct = min(100.0, start_offset / total_records * 100)
            print(f"  Page {draw-1}: scanned {len(page_data):,}, kept {len(kept):,} "
                  f"| kept {saved_count:,} | scanned {seen_count:,}/{total_records:,} ({pct:.1f}%)")
            sys.stdout.flush()

    except KeyboardInterrupt:
        print(f"\n[STOP] Interrupted at offset {start_offset:,}. Progress saved.")
    except Exception as e:
        print(f"\n[ERROR] Fatal error: {e}")
        import traceback
        traceback.print_exc()
        return False
    finally:
        client.close()

    save_checkpoint(checkpoint_file, start_offset, saved_count, total_records)
    file_size = os.path.getsize(output_file) / (1024 * 1024) if os.path.exists(output_file) else 0
    print(f"\n[DONE] kept {saved_count:,} records (scanned {seen_count:,}/{total_records:,})")
    print(f"[FILE] {output_file} ({file_size:.1f} MB)")
    if start_offset >= total_records:
        print("[OK] All pages fetched!")
        os.remove(checkpoint_file)
    return True

In [ ]:
# Run the pull. (Skip this cell if you only want to convert an existing JSONL.)
pull(TAHUN, KLDI, OUTPUT_FILE, CHECKPOINT_FILE)

## 3b. (Optional) Filter an existing full-year Feather by idKldi

If you've already pulled the **full year** (e.g. `sirup_2026_all.feather`), don't re-pull just to get one agency — filter the existing file in seconds. Set `ID_KLDI` above and run this cell. Produces `sirup_YYYY_<KLDI>_all.feather` + `.csv`.

In [ ]:
# Fast path: filter the already-pulled full-year Feather by idKldi (no API call).
import pandas as pd

SOURCE_FEATHER = f"sirup_{TAHUN}_all.feather"   # the full-year file you already have
out_feather = f"sirup_{TAHUN}_{KLDI}_all.feather" if KLDI else f"sirup_{TAHUN}_all.feather"
out_csv = out_feather.replace(".feather", ".csv")

if not KLDI:
    print("[SKIP] ID_KLDI is blank — nothing to filter. Set it in the config cell.")
elif not os.path.exists(SOURCE_FEATHER):
    print(f"[ERROR] {SOURCE_FEATHER} not found. Pull the full year first (leave ID_KLDI blank).")
else:
    df = pd.read_feather(SOURCE_FEATHER)
    sub = df[df["idKldi"].astype(str).str.upper() == KLDI].reset_index(drop=True)
    sub.to_feather(out_feather)
    sub.to_csv(out_csv, sep="|", index=False)
    name = sub["kldi"].iloc[0] if len(sub) else "-"
    print(f"[OK] idKldi={KLDI} ({name}): {len(sub):,} rows")
    print(f"     {out_feather}  ({os.path.getsize(out_feather)/(1024**2):.2f} MB)")
    print(f"     {out_csv}  ({os.path.getsize(out_csv)/(1024**2):.2f} MB)")

## 4. Convert JSONL → Feather + pipe-delimited CSV

In [ ]:
def convert(output_file, feather_file, csv_file):
    """Convert JSONL -> Feather + pipe-delimited CSV using chunked reading."""
    import pandas as pd

    if not os.path.exists(output_file):
        print(f"[ERROR] {output_file} not found. Run the pull cell first.")
        return False

    print(f"[INFO] Reading {output_file} in chunks...")
    chunks = []
    start = time.time()
    chunk_size = 100_000
    count = 0
    with open(output_file, "r", encoding="utf-8") as f:
        chunk = []
        for line in f:
            chunk.append(json.loads(line))
            count += 1
            if len(chunk) >= chunk_size:
                chunks.append(pd.DataFrame(chunk))
                chunk = []
                print(f"  {count:,} records read")
        if chunk:
            chunks.append(pd.DataFrame(chunk))

    print(f"[INFO] Read {count:,} records in {time.time()-start:.1f}s")
    print("[INFO] Concatenating...")
    df = pd.concat(chunks, ignore_index=True)
    del chunks
    print(f"   Shape: {df.shape}")

    print(f"[INFO] Writing {feather_file}...")
    df.to_feather(feather_file)
    feather_mb = os.path.getsize(feather_file) / (1024 ** 2)
    print(f"   Done! Size: {feather_mb:.1f} MB")

    print(f"[INFO] Writing {csv_file} (pipe-delimited)...")
    df.to_csv(csv_file, sep="|", index=False)
    csv_mb = os.path.getsize(csv_file) / (1024 ** 2)
    print(f"   Done! Size: {csv_mb:.1f} MB")

    print(f"[INFO] Total conversion time: {time.time()-start:.1f}s")
    return True

In [ ]:
# Run the conversion. Skipped automatically if NO_CONVERT is True above.
if NO_CONVERT:
    print("[SKIP] NO_CONVERT is True — skipping conversion.")
else:
    convert(OUTPUT_FILE, FEATHER_FILE, CSV_FILE)